In [1]:
!pip install transformers datasets scikit-learn -q

In [2]:
import pandas as pd
import numpy as np
import torch
from torch.nn import CrossEntropyLoss
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from torch.utils.data import Dataset
from sklearn.metrics import classification_report, f1_score

print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA available: True
Device: Tesla T4


In [3]:
train = pd.read_csv("https://raw.githubusercontent.com/HarshAggarwal524/hinglish-sentiment-analysis/main/data/emotion_train_final.csv")
dev = pd.read_csv("https://raw.githubusercontent.com/HarshAggarwal524/hinglish-sentiment-analysis/main/data/emotion_dev_final.csv")

label2id = {'anger': 0, 'joy': 1, 'sadness': 2, 'trust': 3}
id2label = {0: 'anger', 1: 'joy', 2: 'sadness', 3: 'trust'}

train['label'] = train['emotion'].map(label2id)
dev['label'] = dev['emotion'].map(label2id)

print("Train:", train.shape)
print("Dev:", dev.shape)
print("\nTrain distribution:")
print(train['emotion'].value_counts())

Train: (5083, 4)
Dev: (897, 4)

Train distribution:
emotion
anger      2375
joy        1990
sadness     427
trust       291
Name: count, dtype: int64


In [4]:

muril_tokenizer = AutoTokenizer.from_pretrained("google/muril-base-cased")

class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128):
        self.texts = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

train_dataset = EmotionDataset(train, muril_tokenizer)
dev_dataset = EmotionDataset(dev, muril_tokenizer)

print(f"Train dataset: {len(train_dataset)}")
print(f"Dev dataset: {len(dev_dataset)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/411 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/206 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/3.16M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/113 [00:00<?, ?B/s]

Train dataset: 5083
Dev dataset: 897


In [5]:
model = AutoModelForSequenceClassification.from_pretrained(
    "google/muril-base-cased",
    num_labels=4,
    id2label=id2label,
    label2id=label2id
)

# Class weights — inverse frequency to handle imbalance
# anger=2375, joy=1990, sadness=427, trust=291
# Higher weight = more penalty for getting that class wrong
class_weights = torch.tensor([1.0, 1.2, 5.5, 8.0]).to('cuda')

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        from torch.nn import CrossEntropyLoss
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        loss_fct = CrossEntropyLoss(weight=class_weights)
        loss = loss_fct(logits, labels)
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1_macro = f1_score(labels, predictions, average='macro')
    report = classification_report(
        labels, predictions,
        target_names=['anger', 'joy', 'sadness', 'trust'],
        zero_division=0  # suppresses the precision warning
    )
    print(report)
    return {'macro_f1': f1_macro}

training_args = TrainingArguments(
    output_dir="./muril_emotion",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    warmup_steps=200,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=50,
    fp16=True
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    compute_metrics=compute_metrics
)

print("MuRIL ready to train")

pytorch_model.bin:   0%|          | 0.00/953M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w

model.safetensors:   0%|          | 0.00/953M [00:00<?, ?B/s]

MuRIL ready to train


In [6]:
trainer.train()

Epoch,Training Loss,Validation Loss,Macro F1
1,1.300690,1.265762,0.410832
2,1.120603,1.159558,0.453932
3,1.000267,1.085720,0.564980
4,0.892197,1.014298,0.611114
5,0.785195,1.016799,0.601935


              precision    recall  f1-score   support

       anger       0.78      0.91      0.84       419
         joy       0.75      0.87      0.81       351
     sadness       0.00      0.00      0.00        75
       trust       0.00      0.00      0.00        52

    accuracy                           0.76       897
   macro avg       0.38      0.44      0.41       897
weighted avg       0.66      0.76      0.71       897



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

       anger       0.88      0.76      0.82       419
         joy       0.69      0.93      0.79       351
     sadness       0.25      0.17      0.20        75
       trust       0.00      0.00      0.00        52

    accuracy                           0.74       897
   macro avg       0.45      0.47      0.45       897
weighted avg       0.70      0.74      0.71       897



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

              precision    recall  f1-score   support

       anger       0.89      0.82      0.85       419
         joy       0.82      0.82      0.82       351
     sadness       0.28      0.37      0.32        75
       trust       0.42      0.48      0.45        52

    accuracy                           0.76       897
   macro avg       0.60      0.62      0.61       897
weighted avg       0.78      0.76      0.77       897



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

              precision    recall  f1-score   support

       anger       0.89      0.83      0.86       419
         joy       0.81      0.84      0.82       351
     sadness       0.28      0.31      0.29        75
       trust       0.40      0.48      0.43        52

    accuracy                           0.77       897
   macro avg       0.59      0.61      0.60       897
weighted avg       0.78      0.77      0.77       897



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1590, training_loss=1.055836589981175, metrics={'train_runtime': 554.0302, 'train_samples_per_second': 45.873, 'train_steps_per_second': 2.87, 'total_flos': 1671771887784960.0, 'train_loss': 1.055836589981175, 'epoch': 5.0})

In [7]:
results = trainer.evaluate()
print(f"\nMuRIL Best Macro F1: {results['eval_macro_f1']:.3f}")

              precision    recall  f1-score   support

       anger       0.89      0.82      0.85       419
         joy       0.82      0.82      0.82       351
     sadness       0.28      0.37      0.32        75
       trust       0.42      0.48      0.45        52

    accuracy                           0.76       897
   macro avg       0.60      0.62      0.61       897
weighted avg       0.78      0.76      0.77       897



Training Loss,Validation Loss,Epoch,Macro F1
0.785195,1.014298,5,0.611114



MuRIL Best Macro F1: 0.611
